# fuzzy-search exploration

Try `FuzzyPhraseSearcher` from the [`fuzzy-search`](https://github.com/marijnkoolen/fuzzy-search)
package as an alternative to the manual Levenshtein loop in `pattern_merge.py`.

The library was built for HTR text from Dutch archival sources — the same corpus we work with.

**Three things to test:**
1. Does it recover the same split-half → neighbor matches as the current scorer?
2. How does speed compare?
3. Can the distractor mechanism suppress compound-name false positives?


In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib inline


In [2]:
# Install if needed
import importlib
if importlib.util.find_spec('fuzzy_search') is None:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'fuzzy-search'])

import fuzzy_search
print('fuzzy_search version:', fuzzy_search.__version__)


fuzzy_search version: 2.7.1


In [3]:
import sys, os, time
sys.path.insert(0, os.path.abspath(''))

import pandas as pd
from utils import (
    source_mtimes, load_data, enrich_persons_from_abbrd,
    build_merged, load_corrections, load_remappings, apply_corrections,
    load_pattern_synonyms,
)
from pattern_merge import build_anchor_table, _norm, _lev_dist, _strip_infix

t0 = time.perf_counter()
df_persons, df_occurrences, df_abbrd = load_data(source_mtimes())
df_persons_enriched, _ = enrich_persons_from_abbrd(df_persons, df_abbrd)
df_merged, _, _, _ = build_merged(
    df_persons_enriched, df_occurrences, df_abbrd,
    None, load_remappings(),
)
df_merged = apply_corrections(df_merged, load_corrections())
at = build_anchor_table(df_merged, synonyms=load_pattern_synonyms())
print(f'data loaded  {(time.perf_counter()-t0)*1000:.0f} ms')
print(f'{len(at.anchors)} delegates with anchors')


2026-04-21 08:51:22.052 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-04-21 08:51:22.053 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-04-21 08:51:22.054 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-04-21 08:51:22.055 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-04-21 08:51:22.055 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-04-21 08:51:22.055 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-04-21 08:51:22.056 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-04-21 08:51:22.056 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-04-

data loaded  8679 ms
771 delegates with anchors


In [4]:
# ── Correction statistics (all three tiers) ─────────────────────────────
from utils import load_corrections, load_staged_corrections, load_approved_corrections

corr_active   = load_corrections()           # {int(row_idx): {to_id, from_id, ...}}
corr_approved = load_approved_corrections()  # {int(row_idx): {to_id, approved_at, ...}}
corr_staged   = load_staged_corrections()    # {int(row_idx): int(to_id)}

# Helper: extract to_id regardless of value type
def _to_id(v):
    if isinstance(v, dict):
        return str(v.get('to_id', v.get('to_id', '')))
    return str(v)

# For each tier: which row indices exist in df_merged?
idx_set = set(df_merged.index)

def tier_stats(name, d):
    total   = len(d)
    applied = {k: v for k, v in d.items() if k in idx_set}
    stale   = total - len(applied)
    from_ids = df_merged.loc[list(applied.keys()), 'delegate_id'].tolist() if applied else []
    to_ids   = [_to_id(v) for v in applied.values()]
    n_unique_from = len(set(from_ids))
    n_unique_to   = len(set(to_ids))
    print(f'  {name:<12}: {total:>6} total  |  {len(applied):>6} applied  |  {stale:>6} stale  |  {n_unique_from:>5} unique from-ids  |  {n_unique_to:>5} unique to-ids')
    return applied, from_ids, to_ids

print('Correction file statistics')
print('-' * 90)
ap_active,   fi_active,   ti_active   = tier_stats('active',   corr_active)
ap_approved, fi_approved, ti_approved = tier_stats('approved', corr_approved)
ap_staged,   fi_staged,   ti_staged   = tier_stats('staged',   corr_staged)
print('-' * 90)

# Combined (union, active > approved > staged for overlapping keys)
combined = {**corr_staged, **corr_approved, **corr_active}
ap_all = {k: v for k, v in combined.items() if k in idx_set}
print(f'  {"COMBINED":<12}: {len(combined):>6} total  |  {len(ap_all):>6} applied')
print()

# Per-(from_id → to_id) breakdown for the combined set
import pandas as pd
rows = []
for k, v in ap_all.items():
    rows.append({'row_idx': k,
                 'from_id': df_merged.at[k, 'delegate_id'],
                 'to_id':   _to_id(v)})
chg = pd.DataFrame(rows)
summary_chg = (
    chg.groupby(['from_id', 'to_id'])
       .agg(n_rows=('row_idx', 'count'))
       .reset_index()
       .sort_values('n_rows', ascending=False)
)
print(f'Top 20 from_id → to_id remappings (by row count):')
print(summary_chg.head(20).to_string(index=False))


Correction file statistics
------------------------------------------------------------------------------------------
  active      :    135 total  |     135 applied  |       0 stale  |      7 unique from-ids  |      7 unique to-ids
  approved    :  22607 total  |   22606 applied  |       1 stale  |    100 unique from-ids  |     52 unique to-ids
  staged      :  75946 total  |   68092 applied  |    7854 stale  |    324 unique from-ids  |   1506 unique to-ids
------------------------------------------------------------------------------------------
  COMBINED    :  79360 total  |   71505 applied

Top 20 from_id → to_id remappings (by row count):
from_id to_id  n_rows
  16274 16307    2321
  14024 14226    1121
  16261 16307     844
  14025 14024     840
  16887 14226     786
  16217 16189     704
  16307 16189     676
  14025 16887     625
  13498 15127     613
  16209 16120     529
  15155 19809     492
  13552 15127     468
  16211 16189     461
  16192 16120     460
  15138 19809    

## 1. Basic matching — does it find the right neighbors?

Build a `FuzzyPhraseSearcher` from two neighbor anchors and search for each
split half of a known concat candidate.


In [5]:
from fuzzy_search import FuzzyPhraseSearcher

# Config tuned for short proper names with HTR noise
config = {
    'char_match_threshold': 0.5,
    'ngram_threshold': 0.4,
    'levenshtein_threshold': 0.6,
    'ignorecase': True,
    'max_length_variance': 4,
    'ngram_size': 2,
    'skip_size': 1,
}

known_cases = [
    ('vanden Bempden Van Hoorn', 'vanden Bempden', 'Van Hoorn',
     'vanden bempden', 'van hoorn'),
    ('Changuion Geelvinck',      'Changuion',      'Geelvinck',
     'changuion',      'geelvinck'),
    ('Ubingh Sminia',            'Ubingh',         'Sminia',
     'ubingh',         'sminia'),
    ("d'Ablaing van Giessenburgh", "d'Ablaing",   'van Giessenburgh',
     "d'ablaing",      'van giessenburgh'),
]

print(f"{'concat pattern':<32} {'half':<22} {'best match':<25} {'lev_sim':>8}")
print('-' * 92)

for (concat_pat, l_half, r_half, l_anchor, r_anchor) in known_cases:
    phrase_model = [
        {'phrase': l_anchor, 'label': 'left'},
        {'phrase': r_anchor, 'label': 'right'},
    ]
    searcher = FuzzyPhraseSearcher(config=config, phrase_model=phrase_model)
    for half, expected in [(l_half, l_anchor), (r_half, r_anchor)]:
        half_n = _norm(half)
        matches = searcher.find_matches({'text': half_n, 'id': 'x'})
        if matches:
            best = max(matches, key=lambda m: m.levenshtein_similarity)
            print(f"{concat_pat:<32} {half_n:<22} {best.phrase.phrase_string:<25} {best.levenshtein_similarity:>8.3f}")
        else:
            print(f"{concat_pat:<32} {half_n:<22} {'NO MATCH':<25} {'—':>8}")


concat pattern                   half                   best match                 lev_sim
--------------------------------------------------------------------------------------------
vanden Bempden Van Hoorn         vanden bempden         vanden bempden               1.000
vanden Bempden Van Hoorn         van hoorn              van hoorn                    1.000
Changuion Geelvinck              changuion              changuion                    1.000
Changuion Geelvinck              geelvinck              geelvinck                    1.000
Ubingh Sminia                    ubingh                 ubingh                       1.000
Ubingh Sminia                    sminia                 sminia                       1.000
d'Ablaing van Giessenburgh       d'ablaing              d'ablaing                    1.000
d'Ablaing van Giessenburgh       van giessenburgh       van giessenburgh             1.000


## 2. Speed comparison

Build a searcher from **all** delegate anchors and time a lookup,
vs. the current manual `min(_lev_dist(...))` loop.


In [6]:
all_phrases = [
    {'phrase': anchor, 'label': did}
    for did, anchor in at.anchors.items() if anchor
]
print(f'Phrase model size: {len(all_phrases)} anchors')

t0 = time.perf_counter()
searcher_all = FuzzyPhraseSearcher(config=config, phrase_model=all_phrases)
print(f'FuzzyPhraseSearcher built in {(time.perf_counter()-t0)*1000:.0f} ms')


Phrase model size: 771 anchors
FuzzyPhraseSearcher built in 2746 ms


In [7]:
import random

split_halves = []
for (_, l_half, r_half, _, _) in known_cases:
    split_halves += [_norm(l_half), _norm(r_half)]
random_pats = df_merged['pattern'].dropna().sample(92, random_state=42).map(_norm).tolist()
split_halves += random_pats
print(f'Testing {len(split_halves)} queries')

t0 = time.perf_counter()
for half in split_halves:
    searcher_all.find_matches({'text': half, 'id': 'x'})
t_fuzzy = time.perf_counter() - t0
print(f'fuzzy-search:  {t_fuzzy*1000:.0f} ms  ({t_fuzzy/len(split_halves)*1000:.2f} ms/query)')

anchor_list = list(at.anchors.values())
t0 = time.perf_counter()
for half in split_halves:
    min((_lev_dist(half, a) for a in anchor_list), default=1.0)
t_manual = time.perf_counter() - t0
print(f'manual lev:    {t_manual*1000:.0f} ms  ({t_manual/len(split_halves)*1000:.2f} ms/query)')
print(f'Speedup: {t_manual/t_fuzzy:.1f}x')


Testing 100 queries
fuzzy-search:  282 ms  (2.82 ms/query)
manual lev:    25 ms  (0.25 ms/query)
Speedup: 0.1x


## 3. Distractor test — suppress compound-name false positives

Register the full compound name as a `distractor` for each split-half phrase.
A match is suppressed if the text string is closer to the distractor than to the phrase.


In [8]:
distractor_cases = [
    ("d'ablaing van giessenburgh", "d'ablaing",       'van giessenburgh'),
    ('van der capellen tot den pol', 'van der capellen', 'tot den pol'),
]

config_dist = {**config, 'filter_distractors': True, 'include_variants': True}

for (full, left_h, right_h) in distractor_cases:
    phrase_model = [
        {'phrase': left_h,  'label': 'left',  'distractors': [full]},
        {'phrase': right_h, 'label': 'right', 'distractors': [full]},
    ]
    searcher_d = FuzzyPhraseSearcher(config=config_dist, phrase_model=phrase_model)

    matches = searcher_d.find_matches({'text': full, 'id': 'x'})
    print(f"Input: '{full}'")
    if matches:
        for m in matches:
            print(f"  MATCHED '{m.phrase.phrase_string}'  sim={m.levenshtein_similarity:.3f}  <- should be suppressed")
    else:
        print(f"  suppressed (no matches) as expected")

    for half in [left_h, right_h]:
        mh = searcher_d.find_matches({'text': half, 'id': 'x'})
        best = max(mh, key=lambda m: m.levenshtein_similarity) if mh else None
        if best:
            print(f"  half '{half}' -> '{best.phrase.phrase_string}'  sim={best.levenshtein_similarity:.3f}")
        else:
            print(f"  half '{half}' -> NO MATCH")
    print()


Input: 'd'ablaing van giessenburgh'
  MATCHED 'd'ablaing'  sim=1.000  <- should be suppressed
  MATCHED 'van giessenburgh'  sim=1.000  <- should be suppressed
  half 'd'ablaing' -> 'd'ablaing'  sim=1.000
  half 'van giessenburgh' -> 'van giessenburgh'  sim=1.000

Input: 'van der capellen tot den pol'
  MATCHED 'van der capellen'  sim=1.000  <- should be suppressed
  MATCHED 'tot den pol'  sim=1.000  <- should be suppressed
  half 'van der capellen' -> 'van der capellen'  sim=1.000
  half 'tot den pol' -> 'tot den pol'  sim=1.000



## 4. Conclusions

Fill in after running:

| Test | Result |
|------|--------|
| Correct matches for known concat cases? | — |
| Speed vs manual loop | — |
| Distractor suppression works? | — |

**Decision:** integrate into `pattern_merge.py` / keep manual loop?


## 5. Compound-name pre-check

Before flagging a pattern as a concat candidate, check whether the **whole pattern**
already scores well against any single anchor.  If `min(_lev_dist(pattern, anchor) for anchor in all_anchors) ≤ t_compound`,
the pattern is likely a compound name → skip it.

Test this against `ground_truth_concat.csv`:
- `compound_name` rows should be suppressed (TP for the filter)
- `genuine_concat` rows should survive (should not be suppressed)


In [9]:
import pandas as pd

gt = pd.read_csv('ground_truth_concat.csv')

# All anchor strings from the anchor table
anchor_list = [a for a in at.anchors.values() if a]
anchor_list_bare = [_strip_infix(a) for a in anchor_list]

def compound_score(pattern: str) -> float:
    """Min Levenshtein distance between the full pattern and any anchor."""
    p = _norm(pattern)
    p_bare = _strip_infix(p)
    raw  = min(_lev_dist(p, a) for a in anchor_list)
    bare = min(_lev_dist(p_bare, ab) for ab in anchor_list_bare) if p_bare else raw
    return min(raw, bare)

results = []
for _, row in gt.iterrows():
    score = compound_score(str(row['pattern']))
    results.append({
        'pattern': row['pattern'],
        'label':   row['label'],
        'score':   score,
    })

df_r = pd.DataFrame(results)

print("Score distribution by label:")
print(df_r.groupby('label')['score'].describe().round(3))
print()

# Sweep threshold — maximise suppression of compound_name without hurting genuine_concat
print(f"{'t_compound':<12} {'compound suppressed':>20} {'genuine suppressed':>20}")
print('-' * 55)
for t in [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40]:
    suppressed  = df_r[df_r['score'] <= t]
    n_cmpd      = (suppressed['label'] == 'compound_name').sum()
    n_genuine   = (suppressed['label'] == 'genuine_concat').sum()
    total_cmpd  = (df_r['label'] == 'compound_name').sum()
    total_gen   = (df_r['label'] == 'genuine_concat').sum()
    print(f"{t:<12.2f} {n_cmpd:>8}/{total_cmpd} ({n_cmpd/total_cmpd*100 if total_cmpd else 0:.0f}%){n_genuine:>8}/{total_gen} ({n_genuine/total_gen*100 if total_gen else 0:.0f}%)")


Score distribution by label:
                count   mean    std    min    25%    50%    75%    max
label                                                                 
compound_name     8.0  0.247  0.174  0.000  0.083  0.303  0.393  0.429
genuine_concat   13.0  0.412  0.084  0.278  0.357  0.435  0.465  0.538

t_compound    compound suppressed   genuine suppressed
-------------------------------------------------------
0.05                1/8 (12%)       0/13 (0%)
0.10                3/8 (38%)       0/13 (0%)
0.15                3/8 (38%)       0/13 (0%)
0.20                3/8 (38%)       0/13 (0%)
0.25                4/8 (50%)       0/13 (0%)
0.30                4/8 (50%)       2/13 (15%)
0.35                4/8 (50%)       3/13 (23%)
0.40                7/8 (88%)       6/13 (46%)


In [10]:
# Verify compound pre-check in detect_concat_errors
# Compare candidate counts with and without the filter
from pattern_merge import detect_concat_errors, DEFAULT_T_COMPOUND

kwargs = dict(df_merged=df_merged, at=at,
              province='Holland', year_min=1700, year_max=1800)

no_filter = detect_concat_errors(**kwargs, t_compound=None)
with_filter = detect_concat_errors(**kwargs, t_compound=DEFAULT_T_COMPOUND)
_, rejected = detect_concat_errors(**kwargs, t_compound=DEFAULT_T_COMPOUND, return_rejected=True)

compound_rejected = rejected[rejected['rejection_reason'] == 'compound_name']
print(f"Without compound filter : {len(no_filter)} candidates")
print(f"With t_compound={DEFAULT_T_COMPOUND}      : {len(with_filter)} candidates")
print(f"Suppressed as compound  : {len(compound_rejected)}")
if len(compound_rejected):
    print(compound_rejected[['pattern', 'anchor']].to_string(index=False))


Without compound filter : 8 candidates
With t_compound=0.25      : 0 candidates
Suppressed as compound  : 281
                                              pattern                             anchor
                                           de Fremery                            fremery
                                          de Gyzelaer                           gyzelaar
                                          de Gyselaar                           gyzelaar
                                          de Gyzelaar                           gyzelaar
                                          De Gyzelaer                           gyzelaar
                               van Riemsdyk Verbrugge                          verbrugge
                              van Riems lyk Verbrugge                          verbrugge
                                    Verbrugge Meerman                          verbrugge
                                  Changuion Geelvinck                          geelvinck


In [11]:
# Cross-reference the suppressed patterns against ground_truth_concat.csv
# Any genuine_concat in this list = false negative introduced by the filter

gt_norm = gt.copy()
gt_norm['pattern_norm'] = gt_norm['pattern'].map(lambda p: _norm(str(p)))
cr_norm = compound_rejected.copy()
cr_norm['pattern_norm'] = cr_norm['pattern'].map(lambda p: _norm(str(p)))

merged_check = cr_norm.merge(
    gt_norm[['pattern_norm', 'label', 'notes']],
    on='pattern_norm', how='left'
)

in_gt      = merged_check[merged_check['label'].notna()]
false_negs = merged_check[merged_check['label'] == 'genuine_concat']

print(f"Suppressed patterns found in ground truth : {len(in_gt)}")
print(f"  of which genuine_concat (false negatives): {len(false_negs)}")
if len(in_gt):
    print()
    print(in_gt[['pattern', 'anchor', 'label', 'notes']].to_string(index=False))

print()
print("Suppressed patterns NOT in ground truth (candidates for labeling):")
unlabeled = merged_check[merged_check['label'].isna()][['pattern', 'anchor']]
print(f"  {len(unlabeled)} patterns — sample of up to 20:")
print(unlabeled.head(20).to_string(index=False))


Suppressed patterns found in ground truth : 4
  of which genuine_concat (false negatives): 3

                 pattern         anchor          label      notes
     Changuion Geelvinck      geelvinck genuine_concat        NaN
vanden Bempden Van Hoorn vanden bembden genuine_concat        NaN
     Heynsius Spanbroeck       heynsius  compound_name one person
 van Hoornbeeck Ockersse van hoornbeeck genuine_concat        NaN

Suppressed patterns NOT in ground truth (candidates for labeling):
  277 patterns — sample of up to 20:
                    pattern         anchor
                 de Fremery        fremery
                de Gyzelaer       gyzelaar
                de Gyselaar       gyzelaar
                de Gyzelaar       gyzelaar
                De Gyzelaer       gyzelaar
     van Riemsdyk Verbrugge      verbrugge
    van Riems lyk Verbrugge      verbrugge
          Verbrugge Meerman      verbrugge
         Changuim Geelvinck      geelvinck
                : Cordelois      cordeloi

In [12]:
# Patterns in the ambiguous zone 0.20–0.35 — export train split for labeling
# Stratified sample of N_SAMPLE rows so the full score range is represented
# but the labeling load stays manageable.

import numpy as np

N_SAMPLE = 250  # total before train/test split → ~200 train rows

unique_pats = df_merged['pattern'].dropna().unique()
scores = {p: compound_score(str(p)) for p in unique_pats}

zone = (
    pd.DataFrame({'pattern': list(scores.keys()), 'compound_score': list(scores.values())})
    .query('0.20 <= compound_score <= 0.35')
    .sort_values('compound_score')
    .drop_duplicates('pattern')
    .reset_index(drop=True)
)

# Exclude patterns already in ground truth
already_labeled = set(gt['pattern'].map(lambda p: _norm(str(p))))
zone = zone[~zone['pattern'].map(lambda p: _norm(str(p))).isin(already_labeled)].reset_index(drop=True)

# Stratified sample: pick N_SAMPLE rows proportionally from each score quartile
zone['score_quartile'] = pd.qcut(zone['compound_score'], q=4, labels=False, duplicates='drop')
zone = (
    zone.groupby('score_quartile', group_keys=False)
    .apply(lambda g: g.sample(n=max(1, round(N_SAMPLE * len(g) / len(zone))), random_state=42))
    .reset_index(drop=True)
)

# Stratified 80/20 train/test split within the sample
zone['score_quartile'] = pd.qcut(zone['compound_score'], q=4, labels=False, duplicates='drop')
test_idx = (
    zone.groupby('score_quartile', group_keys=False)
    .apply(lambda g: g.sample(frac=0.2, random_state=42))
    .index
)
zone['split'] = 'train'
zone.loc[test_idx, 'split'] = 'test'
zone = zone.drop(columns='score_quartile')

# Keep only train rows
train = zone[zone['split'] == 'train'].drop(columns='split').reset_index(drop=True)

# Add anchor and delegate info
pat_to_did = df_merged[['pattern','delegate_id']].drop_duplicates('pattern').set_index('pattern')['delegate_id'].astype(str)
train['delegate_id'] = train['pattern'].map(pat_to_did)
did_to_anchor = {str(k): v for k, v in at.anchors.items()}
train['anchor'] = train['delegate_id'].map(did_to_anchor)

# Build delegate name: achternaam first, then tussenvoegsel, voornaam, intitulatie
preferred_order = ['achternaam', 'geslachtsnaam', 'tussenvoegsel', 'voornaam', 'intitulatie']
name_cols = [c for c in preferred_order if c in df_persons_enriched.columns]
df_persons_enriched['delegate_id'] = df_persons_enriched['delegate_id'].astype(str)
name_map = (
    df_persons_enriched[['delegate_id'] + name_cols]
    .drop_duplicates('delegate_id')
    .set_index('delegate_id')
    .apply(lambda r: ' '.join(str(v) for v in r if pd.notna(v) and str(v) not in ('','nan')), axis=1)
)
train['delegate_name'] = train['delegate_id'].map(name_map)

# Label columns (fill in 1 for the correct label)
# concat labels:   genuine_concat, compound_name, false_positive
# fragment labels: fragmented_compound (pattern is one name split across two delegates),
#                  compound (multi-word surname, no error)
train['genuine_concat']      = ''
train['compound_name']       = ''
train['fragmented_compound'] = ''
train['compound']            = ''
train['false_positive']      = ''
train['notes']               = ''

col_order = ['pattern','anchor','delegate_id','delegate_name','compound_score',
             'genuine_concat','compound_name','fragmented_compound','compound','false_positive','notes']
train = train[col_order]

out_path = 'ground_truth_concat_candidates.csv'
train.to_csv(out_path, index=False, encoding='utf-8-sig')
print(f"Sampled {len(zone)} from zone, kept {len(train)} train candidates → {out_path}")
print(f"Score range: {train['compound_score'].min():.3f} – {train['compound_score'].max():.3f}")


Sampled 250 from zone, kept 199 train candidates → ground_truth_concat_candidates.csv
Score range: 0.200 – 0.350


/var/folders/9y/s7573g1n3q9fz78c71ryxj4m0000gn/T/ipykernel_86267/700603774.py:28: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(n=max(1, round(N_SAMPLE * len(g) / len(zone))), random_state=42))
/var/folders/9y/s7573g1n3q9fz78c71ryxj4m0000gn/T/ipykernel_86267/700603774.py:36: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(frac=0.2, random_state=42))


In [13]:
# Recovery: restore labels from the Excel-saved (encoding-garbled) file into the freshly
# regenerated clean CSV.  Join on delegate_id (numeric → encoding-safe).
#
# Usage:
#   1. Re-run cell 17 to regenerate a clean ground_truth_concat_candidates.csv
#   2. Run this cell once to patch the labels back in from the garbled file.

GARBLED_PATH = 'ground_truth_concat_candidates_labeled.csv'   # ← rename your Excel save here

label_cols = ['genuine_concat', 'compound_name', 'fragmented_compound', 'compound', 'false_positive', 'notes']

old = pd.read_csv(GARBLED_PATH, dtype={'delegate_id': str}, sep=None, engine='python')
new = pd.read_csv('ground_truth_concat_candidates.csv', dtype={'delegate_id': str})

# Keep only rows where at least one label was filled in
labeled = old[old[label_cols].astype(str).ne('').any(axis=1)][['delegate_id'] + label_cols]

merged = new.drop(columns=label_cols).merge(labeled, on='delegate_id', how='left')
for c in label_cols:
    merged[c] = merged[c].fillna('')

col_order = ['pattern','anchor','delegate_id','delegate_name','compound_score'] + label_cols
merged = merged[col_order]

merged.to_csv('ground_truth_concat_candidates.csv', index=False, encoding='utf-8-sig')
print(f"Patched {len(labeled)} labeled rows back in → ground_truth_concat_candidates.csv")
print(f"Rows with at least one label: {merged[label_cols].astype(str).ne('').any(axis=1).sum()}")


Patched 200 labeled rows back in → ground_truth_concat_candidates.csv
Rows with at least one label: 450


## Plan: compound-name filter validation & threshold tuning

**Goal:** ensure `detect_concat_errors` suppresses compound names (false positives) without dropping genuine concat candidates (false negatives), and export a clean delegates file.

### Steps

1. **Run cell 4** — loads `df_merged`, `at`, `df_persons_enriched` (required by all cells below)

2. **Run cell 20** (threshold sweep)
   - On first run: computes and saves `compound_score_own` + `length_ratio` to the CSV
   - Output: two sweep tables (own_score, length_ratio) with precision / recall / F1

3. **Label more rows** *(if eval set is too small)*
   - Open `ground_truth_concat_candidates.csv` in Excel
   - Mark `x` in the correct column per row (`compound_name`, `genuine_concat`, `fragmented_compound`, `compound`, or `false_positive`)
   - Save as `ground_truth_concat_candidates_labeled.csv`
   - Re-run cell 19 (label recovery) → re-run cell 20

4. **Update thresholds if sweep recommends it**
   - Edit `DEFAULT_T_COMPOUND` and/or `DEFAULT_MIN_LEN_RATIO` in `pattern_merge.py`
   - Re-run cell 13 (`detect_concat_errors` comparison) and cell 14 (cross-reference vs `ground_truth_concat.csv`) to verify no new false negatives

5. **Re-run the delegates export** (last cell)
   - Regenerates `uq_delegates_baked_<date>.parquet` + `.csv` with corrected delegate assignments


In [14]:
# ── Threshold evaluation on labeled ground_truth_concat_candidates.csv ──
# compound_name  = positive class  (what the filter should suppress)
# genuine_concat = must survive    (false negative if suppressed)
#
# Sweep 1: compound_score  – global min-Lev to any anchor.
#          This is what detect_concat_errors uses via DEFAULT_T_COMPOUND.
# Sweep 2: length_ratio    – len(pattern) / len(own anchor).
#          Used by the min-len-ratio guard (DEFAULT_MIN_LEN_RATIO).

import pandas as pd
from pattern_merge import DEFAULT_T_COMPOUND, DEFAULT_MIN_LEN_RATIO

def _read_clean(path):
    df = pd.read_csv(path, dtype={'delegate_id': str}, sep=None, engine='python')
    df.columns = [c.lstrip('\ufeff') for c in df.columns]
    return df

train = _read_clean('ground_truth_concat_candidates.csv')

label_cols = ['genuine_concat', 'compound_name', 'fragmented_compound', 'compound', 'false_positive']
labeled = train[train[label_cols].astype(str).apply(lambda c: c.str.strip() == 'x').any(axis=1)].copy()
print(f'Labeled rows: {len(labeled)} / {len(train)}')
print('Label distribution:')
for c in label_cols:
    n = (labeled[c].astype(str).str.strip() == 'x').sum()
    if n:
        print(f'  {c:<25}: {n}')

# ── Score distributions per label ────────────────────────────────────────────
print('\nScore distributions per label:')
print(f'  {"label":<25} {"n":>5}  {"cmp_score mean/min/max":>30}  {"len_ratio mean/min/max":>30}')
for c in label_cols:
    mask = labeled[c].astype(str).str.strip() == 'x'
    if mask.sum() == 0:
        continue
    cs = labeled.loc[mask, 'compound_score']
    lr = labeled.loc[mask, 'length_ratio'] if 'length_ratio' in labeled.columns else None
    lr_str = f'{lr.mean():.3f} / {lr.min():.3f} / {lr.max():.3f}' if lr is not None else '—'
    print(f'  {c:<25} {mask.sum():>5}  {cs.mean():.3f} / {cs.min():.3f} / {cs.max():.3f}              {lr_str}')

# One label per row via majority vote
vote_cols = [c + '_vote' for c in label_cols]
for c in label_cols:
    labeled[c + '_vote'] = (labeled[c].astype(str).str.strip() == 'x').astype(int)
agg = labeled.groupby('pattern').agg(
    compound_score=('compound_score', 'first'),
    length_ratio=('length_ratio', 'first') if 'length_ratio' in labeled.columns else ('compound_score', 'first'),
    **{c: (c, 'sum') for c in vote_cols}
).reset_index()
agg['majority_label'] = agg[vote_cols].idxmax(axis=1).str.replace('_vote', '')
agg['total_votes'] = agg[vote_cols].sum(axis=1)
eval_df = agg[agg['total_votes'] > 0].copy()
eval_df['_class'] = (eval_df['majority_label'] == 'compound_name').astype(int)
print(f'\nEval set: {len(eval_df)} unique patterns  '
      f'({eval_df["_class"].sum()} compound_name, {(eval_df["_class"]==0).sum()} other)')

# ── Sweep 1: compound_score (global min-Lev — used by DEFAULT_T_COMPOUND) ─────
print(f'\n── Sweep 1: compound_score ≤ t  (DEFAULT_T_COMPOUND currently {DEFAULT_T_COMPOUND}) ──')
print(f'{"t":<8} {"TP":>5} {"FP":>5} {"FN":>5} {"TN":>5}  {"prec":>7} {"rec":>7} {"F1":>7}')
print('-' * 60)
best_f1, best_t = 0.0, None
for t in [round(x * 0.025, 3) for x in range(7, 15)]:
    supp = eval_df['compound_score'] <= t
    TP = int(( supp & (eval_df['_class'] == 1)).sum())
    FP = int(( supp & (eval_df['_class'] == 0)).sum())
    FN = int((~supp & (eval_df['_class'] == 1)).sum())
    TN = int((~supp & (eval_df['_class'] == 0)).sum())
    prec = TP/(TP+FP) if (TP+FP) else 0.0
    rec  = TP/(TP+FN) if (TP+FN) else 0.0
    f1   = 2*prec*rec/(prec+rec) if (prec+rec) else 0.0
    marker = '  ← current' if abs(t - DEFAULT_T_COMPOUND) < 0.001 else ''
    if f1 > best_f1:
        best_f1, best_t = f1, t
    print(f'{t:<8.3f} {TP:>5} {FP:>5} {FN:>5} {TN:>5}  {prec:>7.3f} {rec:>7.3f} {f1:>7.3f}{marker}')
print(f'\nBest F1 {best_f1:.3f} at t={best_t}')
if best_t is not None and abs(best_t - DEFAULT_T_COMPOUND) > 0.001:
    print(f'→ consider changing DEFAULT_T_COMPOUND: {DEFAULT_T_COMPOUND} → {best_t}')
else:
    print(f'→ DEFAULT_T_COMPOUND={DEFAULT_T_COMPOUND} is already at or near optimum')

# ── Sweep 2: length_ratio < t  (DEFAULT_MIN_LEN_RATIO currently) ─────────────
if 'length_ratio' in eval_df.columns:
    print(f'\n── Sweep 2: length_ratio < t  (DEFAULT_MIN_LEN_RATIO currently {DEFAULT_MIN_LEN_RATIO}) ──')
    print(f'{"t":<8} {"TP":>5} {"FP":>5} {"FN":>5} {"TN":>5}  {"prec":>7} {"rec":>7} {"F1":>7}')
    print('-' * 60)
    best_f1_lr, best_t_lr = 0.0, None
    for t in [round(x * 0.1, 1) for x in range(8, 20)]:
        supp = eval_df['length_ratio'] < t
        TP = int(( supp & (eval_df['_class'] == 1)).sum())
        FP = int(( supp & (eval_df['_class'] == 0)).sum())
        FN = int((~supp & (eval_df['_class'] == 1)).sum())
        TN = int((~supp & (eval_df['_class'] == 0)).sum())
        prec = TP/(TP+FP) if (TP+FP) else 0.0
        rec  = TP/(TP+FN) if (TP+FN) else 0.0
        f1   = 2*prec*rec/(prec+rec) if (prec+rec) else 0.0
        marker = '  ← current' if abs(t - DEFAULT_MIN_LEN_RATIO) < 0.001 else ''
        if f1 > best_f1_lr:
            best_f1_lr, best_t_lr = f1, t
        print(f'{t:<8.1f} {TP:>5} {FP:>5} {FN:>5} {TN:>5}  {prec:>7.3f} {rec:>7.3f} {f1:>7.3f}{marker}')
    print(f'\nBest F1 {best_f1_lr:.3f} at t={best_t_lr}')
    if best_t_lr is not None and abs(best_t_lr - DEFAULT_MIN_LEN_RATIO) > 0.001:
        print(f'→ consider changing DEFAULT_MIN_LEN_RATIO: {DEFAULT_MIN_LEN_RATIO} → {best_t_lr}')
    else:
        print(f'→ DEFAULT_MIN_LEN_RATIO={DEFAULT_MIN_LEN_RATIO} is already at or near optimum')


Labeled rows: 450 / 541
Label distribution:
  genuine_concat           : 16
  compound_name            : 250
  fragmented_compound      : 148
  compound                 : 8
  false_positive           : 28

Score distributions per label:
  label                         n          cmp_score mean/min/max          len_ratio mean/min/max
  genuine_concat               16  0.260 / 0.208 / 0.333              —
  compound_name               250  0.257 / 0.200 / 0.333              —
  fragmented_compound         148  0.258 / 0.200 / 0.343              —
  compound                      8  0.248 / 0.208 / 0.296              —
  false_positive               28  0.257 / 0.200 / 0.333              —

Eval set: 110 unique patterns  (64 compound_name, 46 other)

── Sweep 1: compound_score ≤ t  (DEFAULT_T_COMPOUND currently 0.25) ──
t           TP    FP    FN    TN     prec     rec      F1
------------------------------------------------------------
0.175        0     0    64    46    0.000   0.000   0

## 6. Suspicious-pattern review & correction

Identify patterns that are likely concat errors directly from `uq_delegates_baked`:

1. **Length check** – `length_ratio = len(norm(pattern)) / len(norm(anchor))`. A concat of two names is structurally longer than any single anchor.
2. **Neighbor check** – for each suspicious pattern, find same-day neighbors in the occurrences file and test whether the *right half* of the pattern fuzzy-matches a neighbor's anchor.

Output: `suspicious_patterns_review.csv` — review and annotate with `to_delegate_id`, then run the correction cell.


In [15]:
# ── Isolate suspicious patterns by length ratio ─────────────────────
# Requires: at, df_merged loaded in cell 4.
# Reads the latest uq_delegates_baked_*.parquet produced by cell 27.
#
# length_ratio = len(norm(pattern)) / len(norm(anchor))
# A genuine single-delegate pattern stays near 1.0.
# A concat of two names is structurally ~2x -> ratio >= 1.5
#
# Excludes:
#   - delegates without a known anchor (ratio is meaningless with no baseline)
#   - patterns already flagged invalid in pattern_status.json
#   - ghost patterns already identified in pattern_synonyms.json

import glob
import pandas as pd
from pattern_merge import _norm, _lev_dist, _strip_infix
from utils import OCCURRENCES_OUTPUT, load_pattern_status, load_pattern_synonyms

LEN_RATIO_THRESHOLD = 1.5   # tune as needed

# ── Build exclusion sets from status + synonyms ─────────────────────────
_status = load_pattern_status()
_invalid_patterns: set[tuple[str, str]] = {
    (k.split('|')[0], k.split('|')[1])
    for k, v in _status.items()
    if not v and len(k.split('|')) >= 2
}
print(f'Known-invalid (did, pattern) pairs from pattern_status: {len(_invalid_patterns)}')

_synonyms = load_pattern_synonyms()
_ghost_patterns: set[tuple[str, str]] = set()
for _syn in _synonyms:
    _did  = str(_syn.get('delegate_id', ''))
    _pats = _syn.get('patterns', [])
    _fa   = int(_syn.get('freq_a', 0))
    _fb   = int(_syn.get('freq_b', 0))
    if len(_pats) == 2:
        _ghost = str(_pats[0]) if _fa <= _fb else str(_pats[1])
        _ghost_patterns.add((_did, _ghost))
print(f'Known-ghost (did, pattern) pairs from pattern_synonyms: {len(_ghost_patterns)}')

# ── Load latest baked file ──────────────────────────────────────────────
baked_files = sorted(glob.glob(str(OCCURRENCES_OUTPUT.parent / 'uq_delegates_baked_*.parquet')))
if not baked_files:
    raise FileNotFoundError('No uq_delegates_baked_*.parquet found -- run cell 27 first.')
df_baked = pd.read_parquet(baked_files[-1])
df_baked['delegate_id'] = df_baked['delegate_id'].astype(str)
print(f'Loaded {baked_files[-1]}  ({len(df_baked)} delegates)')

did_to_anchor = {str(k): v for k, v in at.anchors.items()}
_no_anchor_skipped = 0

# ── Explode semicolon-separated patterns ────────────────────────────────
rows = []
for _, drow in df_baked.iterrows():
    did    = drow['delegate_id']
    anchor = did_to_anchor.get(did, '')
    if not anchor:
        _no_anchor_skipped += 1
        continue   # ratio meaningless without a baseline
    a_norm = _norm(anchor)
    for pat in str(drow.get('patterns') or drow.get('pattern') or '').split(';'):
        pat = pat.strip()
        if not pat:
            continue
        if (did, pat) in _invalid_patterns or (did, pat) in _ghost_patterns:
            continue
        ratio = len(_norm(pat)) / max(len(a_norm), 1)
        rows.append({
            'delegate_id':   did,
            'delegate_name': str(drow.get('fullname', '')),
            'anchor':        anchor,
            'pattern':       pat,
            'length_ratio':  round(ratio, 3),
        })

df_pats = pd.DataFrame(rows)
print(f'Skipped {_no_anchor_skipped} delegates with no anchor (ratio would be meaningless)')
print(f'Total patterns (after exclusions): {len(df_pats):,}  across {df_pats["delegate_id"].nunique()} delegates')
print(df_pats['length_ratio'].describe().round(3).to_string())

df_susp = (
    df_pats[df_pats['length_ratio'] >= LEN_RATIO_THRESHOLD]
    .sort_values('length_ratio', ascending=False)
    .reset_index(drop=True)
)
print(f'\nSuspicious (ratio >= {LEN_RATIO_THRESHOLD}): {len(df_susp)} patterns '
      f'across {df_susp["delegate_id"].nunique()} delegates')
print(df_susp[['delegate_name', 'anchor', 'pattern', 'length_ratio']].head(40).to_string(index=False))


Known-invalid (did, pattern) pairs from pattern_status: 35
Known-ghost (did, pattern) pairs from pattern_synonyms: 51
Loaded /Users/rikhoekstra/develop/streamlit_worksheet/uq_delegates_baked_20260420.parquet  (812 delegates)
Total patterns (after exclusions): 7,338  across 781 delegates
count    7338.000
mean        1.873
std         4.714
min         0.086
25%         1.000
50%         1.000
75%         1.113
max       257.000

Suspicious (ratio >= 1.5): 1168 patterns across 254 delegates
                                                                       delegate_name                anchor                                                                                                                                                                                                                                                                                                                                                                                                               

In [16]:
# ── Enrich suspicious patterns with neighbor context ─────────────────────
# For each suspicious pattern, find delegates that ever sat near the same
# delegate (within +-WINDOW rank positions on the same day) and check whether
# the right half of the pattern fuzzy-matches a neighbor's anchor.
#
# Left-half infix check: if the left half reduces to nothing after stripping
# Dutch/French infixes (van, de, d', ...), the pattern is just the delegate's
# own name with a tussenvoegsel prefix -- not a concat.  These rows are kept
# in the CSV but flagged left_is_infix=True and sorted to the bottom.
#
# df_susp must exist from the cell above.

from collections import Counter
import pandas as pd
from pattern_merge import _norm, _lev_dist, _strip_infix

WINDOW       = 5      # rank positions either side
HALF_MATCH_T = 0.35   # lev-distance threshold for right-half -> neighbor anchor

# ── Name lookup for neighbors ─────────────────────────────────────────
# df_persons_enriched is loaded in cell 4 alongside df_merged
_name_col = next(
    (c for c in ('fullname', 'naam', 'name', 'delegate_name')
     if c in df_persons_enriched.columns),
    None,
)
if _name_col:
    did_to_name = df_persons_enriched.set_index('delegate_id')[_name_col].astype(str).to_dict()
else:
    did_to_name = {}
print(f'Name lookup built: {len(did_to_name)} entries')

# ── Build global neighbor table once ───────────────────────────────────
dp = at.day_pos.copy()
dp['delegate_id'] = dp['delegate_id'].astype(str)

neighbor_counts = {}   # {delegate_id: Counter({neighbor_id: n_days_co_occurring})}

for day, day_grp in dp.groupby('_day'):
    day_grp = day_grp.sort_values('_rank')
    ids   = day_grp['delegate_id'].tolist()
    ranks = day_grp['_rank'].tolist()
    for i, (did, rank) in enumerate(zip(ids, ranks)):
        nbrs = [
            ids[j] for j in range(len(ids))
            if j != i and abs(ranks[j] - rank) <= WINDOW
        ]
        if did not in neighbor_counts:
            neighbor_counts[did] = Counter()
        neighbor_counts[did].update(nbrs)

print(f'Global neighbor table built: {len(neighbor_counts)} delegates')

# ── Anchor lookup ───────────────────────────────────────────────────────
anchor_list_did = [(str(did), anc) for did, anc in at.anchors.items() if anc]

def _halves(pat_norm: str) -> tuple[str, str]:
    words = pat_norm.split()
    mid = max(1, len(words) // 2)
    return ' '.join(words[:mid]), ' '.join(words[mid:])

def _left_is_infix(left: str) -> bool:
    """Return True if left half is nothing but Dutch/French infixes."""
    return len(_strip_infix(left).strip()) < 2

def _best_neighbor_match(right_half: str, neighbor_ids: set) -> tuple:
    best_score, best_did, best_anchor = 1.0, None, None
    rh_bare = _strip_infix(right_half)
    for ndid, nanc in anchor_list_did:
        if ndid not in neighbor_ids:
            continue
        s  = _lev_dist(right_half, nanc)
        sb = _lev_dist(rh_bare, _strip_infix(nanc)) if rh_bare else s
        score = min(s, sb)
        if score < best_score:
            best_score, best_did, best_anchor = score, ndid, nanc
    return best_did, best_anchor, round(best_score, 3)

# ── Enrich each suspicious pattern ─────────────────────────────────────────
results = []
for _, srow in df_susp.iterrows():
    did    = srow['delegate_id']
    pat    = srow['pattern']
    nbr_counter = neighbor_counts.get(did, Counter())
    neighbor_ids = set(nbr_counter.keys())

    left, rh = _halves(_norm(pat))
    infix_only = _left_is_infix(left)
    nb_did, nb_anchor, nb_score = _best_neighbor_match(rh, neighbor_ids)
    nb_name = did_to_name.get(nb_did, '') if nb_did else ''

    results.append({
        'delegate_id':        did,
        'delegate_name':      srow['delegate_name'],
        'anchor':             srow['anchor'],
        'pattern':            pat,
        'length_ratio':       srow['length_ratio'],
        'left_half':          left,
        'left_is_infix':      infix_only,
        'right_half':         rh,
        'neighbor_did':       nb_did,
        'neighbor_name':      nb_name,
        'neighbor_anchor':    nb_anchor,
        'neighbor_score':     nb_score,
        'n_distinct_neighbors': len(neighbor_ids),
        'to_delegate_id':     '',
        'notes':              '',
    })

df_review = pd.DataFrame(results)

# Sort: non-infix first (actionable), then by score asc / ratio desc
df_review = df_review.sort_values(
    ['left_is_infix', 'neighbor_score', 'length_ratio'],
    ascending=[True, True, False],
).reset_index(drop=True)

real    = df_review[~df_review['left_is_infix']]
infix   = df_review[df_review['left_is_infix']]
strong  = real[real['neighbor_score'] <= HALF_MATCH_T]
print(f'Suspicious patterns total:          {len(df_review)}')
print(f'  left-is-infix (filtered to bottom): {len(infix)}')
print(f'  actionable:                         {len(real)}')
print(f'  strong neighbor match (<={HALF_MATCH_T}):   {len(strong)}')
print()
print(strong[
    ['delegate_name', 'anchor', 'pattern', 'length_ratio',
     'left_half', 'right_half', 'neighbor_name', 'neighbor_anchor', 'neighbor_score']
].head(40).to_string(index=False))

out_csv = 'suspicious_patterns_review.csv'
df_review.to_csv(out_csv, index=False, encoding='utf-8-sig')
print(f'\nSaved -> {out_csv}  ({len(df_review)} rows)')
print('Fill in `to_delegate_id` for confirmed concat errors, then run the correction cell.')


Global neighbor table built: 875 delegates
Suspicious patterns:              1168
  strong neighbor match (≤0.35): 326

                                                                                     delegate_name        anchor                           pattern  length_ratio       neighbor_anchor  neighbor_score
                                                                 Wassenaer Obdam, Carel George van                           Van Wassenaer Catwyck        21.000 van wassenaer catwyck             0.0
                                                                 Wassenaer Obdam, Carel George van                           van Wassenaer Catwyck        21.000 van wassenaer catwyck             0.0
Wassenaer, Frederik Hendrik vanbeide Katwijken en 't Zand en Valkenburg, Rijnsaterwoude, Raaphorst                               van van Wassenaer        17.000         van wassenaer             0.0
                                                               Sirtema van Grovestin

In [17]:
# ── Stage corrections from suspicious_patterns_review.csv ──────────────────
# Read the annotated review CSV and write confirmed entries into
# staged_corrections.json.  Only rows where to_delegate_id is filled in
# and the pattern actually appears in df_merged are staged.
#
# Format written: {row_index: to_delegate_id, ...}  (same as existing staged file)
# Merges with any existing staged corrections (does not overwrite them).

import json as _json
import pandas as pd
from pathlib import Path
from utils import load_staged_corrections

REVIEW_CSV  = 'suspicious_patterns_review.csv'
STAGED_PATH = Path('staged_corrections.json')

# ── Load review file ────────────────────────────────────────────────
df_rev = pd.read_csv(REVIEW_CSV, dtype={'delegate_id': str, 'to_delegate_id': str})
df_rev.columns = [c.lstrip('\ufeff') for c in df_rev.columns]

confirmed = df_rev[
    df_rev['to_delegate_id'].notna() &
    (df_rev['to_delegate_id'].str.strip() != '')
].copy()
confirmed['to_delegate_id'] = confirmed['to_delegate_id'].str.strip()
print(f'Confirmed corrections in review file: {len(confirmed)}')

# ── Map (delegate_id, pattern) → row indices in df_merged ───────────────
new_corrections = {}   # {row_index: to_delegate_id}
skipped = []

for _, crow in confirmed.iterrows():
    did    = str(crow['delegate_id'])
    pat    = str(crow['pattern'])
    to_did = str(crow['to_delegate_id'])

    mask = (
        (df_merged['delegate_id'].astype(str) == did) &
        (df_merged['pattern'] == pat)
    )
    row_idxs = df_merged.index[mask].tolist()

    if not row_idxs:
        skipped.append((did, pat, 'not found in df_merged'))
        continue

    for ridx in row_idxs:
        new_corrections[ridx] = to_did

print(f'Row indices to stage: {len(new_corrections)}')
if skipped:
    print(f'Skipped (not in df_merged): {len(skipped)}')
    for did, pat, reason in skipped[:10]:
        print(f'  delegate {did}  pattern={repr(pat)}  ({reason})')

# ── Merge with existing staged corrections ────────────────────────────
existing = {int(k): v for k, v in load_staged_corrections().items()}
merged_corrections = {**existing, **{int(k): v for k, v in new_corrections.items()}}

STAGED_PATH.write_text(
    _json.dumps({str(k): v for k, v in sorted(merged_corrections.items())},
                ensure_ascii=False, indent=2),
    encoding='utf-8',
)
print(f'\nStaged corrections written -> {STAGED_PATH}')
print(f'  previously staged : {len(existing)}')
print(f'  newly added       : {len(new_corrections)}')
print(f'  total             : {len(merged_corrections)}')
print('\nRe-run the baked export cell to apply these corrections.')


Confirmed corrections in review file: 0
Row indices to stage: 0

Staged corrections written -> staged_corrections.json
  previously staged : 75946
  newly added       : 0
  total             : 75946

Re-run the baked export cell to apply these corrections.


## 7. Build & export unique-delegates file

Apply all three correction tiers, coalesce deficient fields from MySQL,
add delegates absent from parquet, build a `patterns` field, and save
`uq_delegates_baked_<date>.parquet` + `.csv`.

Run this after any threshold or correction change is finalised.


In [18]:
# ── Build updated unique-delegates file (active-only + MySQL supplement) ────
# Strategy:
#   1. Apply all three correction tiers → active_ids.
#   2. Keep only active delegates from df_persons_enriched.
#   3. Fetch MySQL persoon data for all active_ids.
#   4. Coalesce deficient fields (geslachtsnaam, voornaam, fullname,
#      geboortejaar, overlijdensjaar) from MySQL where parquet has nulls.
#      (provincie not in MySQL persoon — cannot supplement from there.)
#   5. Add active delegates entirely absent from parquet (from MySQL).
#   6. Build a "patterns" field: all occurrence patterns per delegate.
#   7. Save active-only frame.

import datetime
import pymysql
import pandas as pd
from utils import (
    apply_corrections, load_corrections,
    load_staged_corrections, load_approved_corrections,
    OCCURRENCES_OUTPUT,
)

CONN = dict(host='localhost', user='rik', password='X0chi', charset='utf8mb4')

# ── 1. Corrections → active_ids ───────────────────────────────────────────────
df_baked = apply_corrections(df_merged, load_staged_corrections())
df_baked = apply_corrections(df_baked,  load_approved_corrections())
df_baked = apply_corrections(df_baked,  load_corrections())
df_baked['delegate_id'] = df_baked['delegate_id'].astype(str)
active_ids = set(df_baked['delegate_id'].unique())
print(f'Active delegate_ids after corrections: {len(active_ids)}')

# ── 2. Active parquet persons only ────────────────────────────────────────────
df_persons_baked = df_persons_enriched.copy()
df_persons_baked['delegate_id'] = df_persons_baked['delegate_id'].astype(str)
df_persons_baked = df_persons_baked[
    df_persons_baked['delegate_id'].isin(active_ids)
].copy()
df_persons_baked['active_after_corrections'] = True
print(f'Active persons in parquet: {len(df_persons_baked)}')

# ── 3. Fetch MySQL persoon for all active_ids ─────────────────────────────────
active_ids_int = [int(i) for i in active_ids if str(i).isdigit()]
fmt = ','.join(['%s'] * len(active_ids_int))
with pymysql.connect(db='raa_nw', **CONN) as conn:
    df_mysql = pd.read_sql(
        f'SELECT id, geslachtsnaam, tussenvoegsel, voornaam, '
        f'       geboortejaar, overlijdensjaar, heerlijkheid '
        f'FROM persoon '
        f'WHERE id IN ({fmt}) '
        f'  AND (mark_for_delete = 0 OR mark_for_delete IS NULL)',
        conn,
        params=active_ids_int,
    )
df_mysql['delegate_id'] = df_mysql['id'].astype(str)
df_mysql = df_mysql.drop(columns=['id'])
print(f'MySQL rows fetched for active ids: {len(df_mysql)}')

# ── 4. Coalesce deficient fields from MySQL ───────────────────────────────────
# Columns to fill from MySQL when null/empty in parquet:
SUPPLEMENT_COLS = ['geslachtsnaam', 'tussenvoegsel', 'voornaam',
                   'geboortejaar', 'overlijdensjaar', 'heerlijkheid']

def _is_blank(s: pd.Series) -> pd.Series:
    return s.isna() | s.astype(str).str.strip().isin(['', 'nan', 'None'])

db_cols = {c: f"_db_{c}" for c in SUPPLEMENT_COLS}
df_persons_baked = df_persons_baked.merge(
    df_mysql[['delegate_id'] + SUPPLEMENT_COLS].rename(columns=db_cols),
    on='delegate_id', how='left',
)
for col, db_col in db_cols.items():
    if col in df_persons_baked.columns and db_col in df_persons_baked.columns:
        fill_mask = _is_blank(df_persons_baked[col]) & ~_is_blank(df_persons_baked[db_col])
        df_persons_baked.loc[fill_mask, col] = df_persons_baked.loc[fill_mask, db_col]
df_persons_baked = df_persons_baked.drop(columns=list(db_cols.values()), errors='ignore')

# Rebuild fullname where blank after coalescing name parts
if 'fullname' in df_persons_baked.columns:
    blank_fn = _is_blank(df_persons_baked['fullname'])
    if blank_fn.any():
        def _build_fullname(row):
            surname  = str(row.get('geslachtsnaam', '') or '').strip()
            infix    = str(row.get('tussenvoegsel', '') or '').strip()
            forename = str(row.get('voornaam', '') or '').strip()
            rest = ' '.join(p for p in [forename, infix] if p)
            return f"{surname}, {rest}".strip(', ') if surname else rest
        df_persons_baked.loc[blank_fn, "fullname"] = (
            df_persons_baked[blank_fn].apply(_build_fullname, axis=1)
        )

# ── 5. Add active delegates absent from parquet ───────────────────────────────
parquet_ids = set(df_persons_baked['delegate_id'].unique())
missing_ids = active_ids - parquet_ids
print(f'Active delegates not in parquet: {len(missing_ids)}')

if missing_ids:
    df_missing = df_mysql[df_mysql['delegate_id'].isin(missing_ids)].copy()
    df_missing['active_after_corrections'] = True
    if 'cons_id_str' in df_persons_baked.columns:
        df_missing['cons_id_str'] = df_missing['delegate_id']
    df_missing['fullname'] = df_missing.apply(
        lambda r: (f"{r.get('geslachtsnaam','') or ''}, "
                   f"{r.get('voornaam','') or ''}").strip(', '),
        axis=1,
    )
    df_persons_baked = pd.concat(
        [df_persons_baked, df_missing.reindex(columns=df_persons_baked.columns)],
        ignore_index=True,
    )
    print(f'  Added {len(df_missing)} rows from MySQL')

# ── 6. Build 'patterns' field ─────────────────────────────────────────────────
# Collect distinct occurrence patterns per delegate from the baked frame
occ_patterns = (
    df_baked[['delegate_id', 'pattern']]
    .dropna(subset=['pattern'])
    .drop_duplicates()
    .groupby('delegate_id')['pattern']
    .apply(lambda s: ';'.join(sorted(set(s.dropna().astype(str)))))
)
df_persons_baked['patterns'] = df_persons_baked['delegate_id'].map(occ_patterns)
# Fall back to existing 'pattern' column for rows where occ lookup is blank
if 'pattern' in df_persons_baked.columns:
    fallback = _is_blank(df_persons_baked['patterns'])
    df_persons_baked.loc[fallback, 'patterns'] = df_persons_baked.loc[fallback, 'pattern']

# ── 7. Save ───────────────────────────────────────────────────────────────────
today = datetime.date.today().strftime('%Y%m%d')
out_path = OCCURRENCES_OUTPUT.parent / f'uq_delegates_baked_{today}.parquet'
df_persons_baked.to_parquet(out_path, index=False)
print(f'\nSaved → {out_path.name}  ({len(df_persons_baked)} rows, all active)')
print(f'  patterns populated   : {df_persons_baked["patterns"].notna().sum()}')
print(f'  geslachtsnaam blank  : {_is_blank(df_persons_baked["geslachtsnaam"]).sum()}')
print(f'  voornaam blank       : {_is_blank(df_persons_baked["voornaam"]).sum()}')
print(f'  fullname blank       : {_is_blank(df_persons_baked["fullname"]).sum()}')
print(f'  provincie blank      : {_is_blank(df_persons_baked["provincie"]).sum()} (not in MySQL)')
print(f'  geboortejaar blank   : {_is_blank(df_persons_baked["geboortejaar"]).sum()}')
print(f'  overlijdensjaar blank: {_is_blank(df_persons_baked["overlijdensjaar"]).sum()}')

# Also save as CSV for human review / correction
csv_path = out_path.with_suffix(".csv")
df_persons_baked.to_csv(csv_path, index=False, encoding="utf-8-sig")
print(f"Also saved → {csv_path.name}")


Active delegate_ids after corrections: 2267
Active persons in parquet: 792
MySQL rows fetched for active ids: 748
Active delegates not in parquet: 1504
  Added 20 rows from MySQL


/var/folders/9y/s7573g1n3q9fz78c71ryxj4m0000gn/T/ipykernel_86267/2927304006.py:45: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_mysql = pd.read_sql(



Saved → uq_delegates_baked_20260421.parquet  (812 rows, all active)
  patterns populated   : 812
  geslachtsnaam blank  : 1
  voornaam blank       : 2
  fullname blank       : 1
  provincie blank      : 31 (not in MySQL)
  geboortejaar blank   : 158
  overlijdensjaar blank: 111
Also saved → uq_delegates_baked_20260421.csv
